<a href="https://colab.research.google.com/github/SnehaVenkatesh19/1099-Vendor-Reporting/blob/main/1099_Vendor_reporting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install mysql-connector-python
!pip install pandas



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.9/33.9 MB 46.2 MB/s eta 0:00:00


In [4]:
from google.colab import files

uploaded = files.upload()


Saving vendors.csv to vendors (1).csv
Saving payments.csv to payments (1).csv


In [5]:
import mysql.connector

conn = mysql.connector.connect(
    host="add-your-amazon-db-host",
    user="",
    password="",
    database="amazon-db-name"
)

cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS netflix;")


In [6]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS vendors (
    vendor_id VARCHAR(10) PRIMARY KEY,
    vendor_name VARCHAR(255),
    payment_method VARCHAR(50),
    EIN VARCHAR(20)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS payments (
    payment_id VARCHAR(10) PRIMARY KEY,
    vendor_id VARCHAR(10),
    payment_date DATE,
    payment_amount DECIMAL(10, 2),
    production_name VARCHAR(255),
    FOREIGN KEY (vendor_id) REFERENCES vendors(vendor_id)
);
""")

conn.commit()




In [7]:
import pandas as pd
vendors = pd.read_csv("vendors.csv")
payments = pd.read_csv("payments.csv")

for _, row in vendors.iterrows():
    cursor.execute("""
        INSERT INTO vendors (vendor_id, vendor_name, payment_method, EIN)
        VALUES (%s, %s, %s, %s)
    """, tuple(row.fillna("")))

for _, row in payments.iterrows():
    cursor.execute("""
        INSERT INTO payments (payment_id, vendor_id, payment_date, payment_amount, production_name)
        VALUES (%s, %s, %s, %s, %s)
    """, tuple(row))

conn.commit()



In [8]:
query = """
SELECT v.vendor_id, v.vendor_name, v.payment_method, v.EIN,
       SUM(p.payment_amount) AS total_paid
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
GROUP BY v.vendor_id
HAVING total_paid > 600;
"""

df_1099 = pd.read_sql(query, conn)
df_1099.head()


<ipython-input-8-4cb0846649b1>:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_1099 = pd.read_sql(query, conn)


,vendor_id,vendor_name,payment_method,EIN,total_paid
0,V001,"Rodriguez, Figueroa and Sanchez",Wire Transfer,667-48-7178,4087.73
1,V002,Doyle Ltd,Check,,9561.67
2,V003,"Mcclain, Miller and Henderson",Wire Transfer,530-58-1983,7593.86
3,V004,Davis and Sons,Wire Transfer,254-29-1050,4458.29
4,V005,"Guzman, Hoffman and Baldwin",Check,347-03-9639,4220.67


##Monthly Payment Summary by Vendor


In [11]:
query1 = """SELECT
v.vendor_id,
v.vendor_name,
DATE_FORMAT(p.payment_date, '%Y-%m') AS payment_month,
SUM(p.payment_amount) AS total_paid
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
GROUP BY v.vendor_id, payment_month
ORDER BY payment_month, total_paid DESC;"""
df1=pd.read_sql(query1, conn)
df1.head()


<ipython-input-11-f577b15c28df>:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1=pd.read_sql(query1, conn)


,vendor_id,vendor_name,payment_month,total_paid
0,V032,Carlson-Cruz,2024-04,1958.54
1,V048,"Martin, Rose and Obrien",2024-04,1144.09
2,V037,Smith-Bowen,2024-04,1059.16
3,V002,Doyle Ltd,2024-04,764.30
4,V027,Baker and Sons,2024-04,627.83


##Vendors Receiving Only One Payment

In [12]:
query2 = """SELECT v.vendor_id, v.vendor_name, COUNT(p.payment_id) AS payment_count
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
GROUP BY v.vendor_id
HAVING payment_count = 1;
"""
df2=pd.read_sql(query2, conn)
df2.head()

<ipython-input-12-92fccc95c2d2>:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2=pd.read_sql(query2, conn)


,vendor_id,vendor_name,payment_count


##Vendors Paid > $1,000 in a Single Transaction

In [13]:
query3 = """SELECT v.vendor_id, v.vendor_name, p.payment_amount, p.payment_date
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
WHERE p.payment_amount > 1000
ORDER BY p.payment_amount DESC;
"""
df3=pd.read_sql(query3, conn)
df3.head()

<ipython-input-13-e943920f3f04>:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3=pd.read_sql(query3, conn)


,vendor_id,vendor_name,payment_amount,payment_date
0,V032,Carlson-Cruz,1499.59,2024-04-22
1,V023,Perez Inc,1495.12,2024-11-09
2,V036,Rodriguez-Graham,1489.80,2025-03-05
3,V011,Blair PLC,1486.23,2024-12-25
4,V046,Spence PLC,1481.55,2024-08-21


##Average Payment Size per Vendor

In [14]:
query4 = """SELECT v.vendor_id, v.vendor_name,
       COUNT(p.payment_id) AS num_payments,
       SUM(p.payment_amount) AS total_paid,
       AVG(p.payment_amount) AS avg_payment
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
GROUP BY v.vendor_id
ORDER BY avg_payment DESC;
"""
df4=pd.read_sql(query4, conn)
df4.head()

<ipython-input-14-04dd7da2348f>:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4=pd.read_sql(query4, conn)


,vendor_id,vendor_name,num_payments,total_paid,avg_payment
0,V034,Frazier Inc,3,3932.98,1310.993333
1,V029,"Ross, Robinson and Bright",6,7305.94,1217.656667
2,V004,Davis and Sons,4,4458.29,1114.572500
3,V027,Baker and Sons,7,7773.11,1110.444286
4,V005,"Guzman, Hoffman and Baldwin",4,4220.67,1055.167500


##1099 Eligibility with Status Flag

In [15]:
query5 = """SELECT v.vendor_id, v.vendor_name, v.EIN,
       SUM(p.payment_amount) AS total_paid,
       CASE
           WHEN SUM(p.payment_amount) > 600 THEN 'Eligible'
           ELSE 'Not Eligible'
       END AS reporting_status
FROM vendors v
JOIN payments p ON v.vendor_id = p.vendor_id
GROUP BY v.vendor_id;
"""
df5=pd.read_sql(query5, conn)
df5.head()

<ipython-input-15-9c6388d8e594>:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df5=pd.read_sql(query5, conn)


,vendor_id,vendor_name,EIN,total_paid,reporting_status
0,V001,"Rodriguez, Figueroa and Sanchez",667-48-7178,4087.73,Eligible
1,V002,Doyle Ltd,,9561.67,Eligible
2,V003,"Mcclain, Miller and Henderson",530-58-1983,7593.86,Eligible
3,V004,Davis and Sons,254-29-1050,4458.29,Eligible
4,V005,"Guzman, Hoffman and Baldwin",347-03-9639,4220.67,Eligible


In [17]:
df1.to_csv("monthly_payments.csv", index=False)
df2.to_csv("one_time_vendors.csv", index=False)
df3.to_csv("large_single_payments.csv", index=False)
df4.to_csv("average_payment_per_vendor.csv", index=False)
df5.to_csv("1099_status_flag.csv", index=False)

from google.colab import files
files.download("monthly_payments.csv")
files.download("one_time_vendors.csv")
files.download("large_single_payments.csv")
files.download("average_payment_per_vendor.csv")
files.download("1099_status_flag.csv")





<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>